In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/noumanarif.personal@gmail.com/Batch_Processing_Data_Engineering/consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")


In [0]:
data_source = dbutils.widgets.get("data_source")
catalog = dbutils.widgets.get("catalog")
print(catalog, data_source)

In [0]:
base_path = f"s3://sportsbar-childcompany-bucket/{data_source}/*.csv"
display(base_path)

In [0]:
df = (
    spark.read.format("csv") 
        .option("header", True) 
        .option("inferSchema", True)
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
    )

    
display(df.limit(7))

In [0]:
df.printSchema()

##Why CDF(Change Data Feed) is Used?

CDF is primarily used for optimizing downstream incremental data pipelines, often moving data from a Bronze layer to a Silver layer. Instead of the Silver pipeline scanning the entire Bronze table to figure out what data is new or modified, it queries the CDF to ask: "Give me only the specific rows that were inserted, updated, or deleted since my last run." This drastically reduces compute costs and processing time.

Because Time Travel is an inherent, automatic feature of the _delta_log, and CDF is a specialized optimization tool for tracking row-level operations, the assumption that .option("delta.enableChangeDataFeed", "true") is what enables Time Travel is incorrect.

In [0]:
# now dump this data on bronze
# .option("delta.enableChangeDataFeed", "true") enables explicit row-level change tracking. While Time Travel(_delta_log) shows you the entire state of the table at a point in the past, CDF shows you the exact events that occurred between two points in time.
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

## How Change Data Feed Functions ?
When CDF is enabled, Delta Lake begins writing extra hidden data files into a _change_data directory alongside your standard Parquet files whenever a MERGE, UPDATE, or DELETE operation occurs. (Standard INSERT and APPEND operations do not require extra files, as Delta just reads the new Parquet files directly).

These CDF files record the exact rows that were modified and append specific metadata columns to them:

_change_type: Indicates if the row was an insert, delete, update_preimage (the old value before the update), or update_postimage (the new value after the update).

_commit_version: The Delta transaction version where the change happened.

_commit_timestamp: The exact time the change occurred.


so in delta lake , delta tables created the actual data of delta tables will be stored in parquet file format where as its metadata is in delta_log where as to track row level changes or tracking the modified rows there is _chage_data directory that tracks the changed / modified rows.

###Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)

In [0]:
# Remove duplicates - deduplication
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count") > 1)
display(df_duplicates)

In [0]:
df_silver = df_bronze.dropDuplicates(["customer_id"])
print("Rows before de-duplication", df_bronze.count())
print("Rows after de-duplication", df_silver.count())

In [0]:
# Trim the spaces - remove the leading spaces in strings
df_silver = df_silver.withColumn("customer_name", F.trim(F.col("customer_name")))
display(df_silver)

In [0]:
df_silver.select("city").distinct().show()

In [0]:
## See typos incorrect spellings
## fix it with proper mapping
city_mapping = {
    'Bengaluruu':'Bengaluru',
    'Bengalore':'Bengaluru',

    'Hyderabadd':'Hyderabad',
    'Hyderbad':'Hyderabad',

    'NewDelhi':'New Delhi',
    'NewDheli':'New Delhi',
    'NewDelhee':'New Delhi'
}

allowed = ['Bengaluru','Hyderabad','New Delhi']
df_silver = (df_silver
            .replace(city_mapping, subset = ["city"])
            .withColumn("city", F.when(F.col("city").isNull(),None)
                                .when(F.col("city").isin(allowed),F.col("city"))
                                .otherwise(None)
        )
)


# Check
df_silver.select("city").distinct().show()


In [0]:
# Fix customer_names inconsistencies
df_silver.select("customer_name").distinct().show()

In [0]:
df_silver = df_silver.withColumn("customer_name", F.when(F.col("customer_name").isNull(),None)
                                .otherwise(F.initcap("customer_name"))
)
df_silver.select("customer_name").distinct().show()


In [0]:
## there are some NULL cities
df_silver.filter(F.col("city").isNull()).show(truncate = False)

In [0]:
# notice that these customer has some cities assigned
# Now engineers team talked to business management team and upon their confirmation we have assigned the cities to the customers

customer_city_fix = {
    789403 : "New Delhi",
    789420 : "Bengaluru",
    789521 : "Hyderabad",
    789603 : "Hyderabad"
}

# Noe create dataframe out of it
df_fix = spark.createDataFrame([(k,v) for k,v in customer_city_fix.items()],
                               ["customer_id", "fixed_city"]
)

display(df_fix)

In [0]:
df_silver =(df_silver
            .join(df_fix, "customer_id","left")
            .withColumn("city", F.coalesce("city", "fixed_city")) ## REplace null with foxed city
            .drop("fixed_city")
            )
display(df_silver)

In [0]:
# Convert customer_id to string
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
display(df_silver.printSchema())

In [0]:
display(df_silver)

In [0]:
# now we have to unify the schema with gold layer's customer_table
df_silver = (
    df_silver
    # Build final customer dolumn as aggregated Customer+city , if city not known then customer+unknown
    .withColumn("customer", F.concat_ws("-","customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    # static atributes aligned with parent model
    # obviously all these columns are filled after meeting with business manager/ team
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
    )


display(df_silver)

In [0]:
## lets write it to silver table
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("merge_schema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

#Gold Processing _ CustomerDimTable

In [0]:
df_silver = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")


# Now unify Schema - take only required columns for gold layer
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")
display(df_gold)

In [0]:
# Write it to gold layer
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")
                 

In [0]:
## unify schema
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel",
)

In [0]:
# perform UPSERT
# meaning id customer_id matched then update else insert
# Schema unification

delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()